# 03. External validation

This notebook performs external validation on AIBL using the reduced shared-variable setting.

The model is trained on ADNI using clinical variables shared between ADNI and AIBL and then evaluated on:

- ADNI held-out test set
- AIBL external cohort

AIBL samples are not used for training, validation, hyperparameter tuning, early stopping, or model selection.

This notebook reuses the model architecture and training utilities, selected hyperparameter configuration, and reproducibility settings defined in `02_model_training_and_internal_validation.ipynb`.

## 1. External validation settings

The external validation model uses the clinical variables available in both ADNI and AIBL.

In [ ]:
ADNI_CSV = "../data/raw/adni_clinical_dataframe.csv"
AIBL_CSV = "../data/raw/aibl_clinical_dataframe.csv"

RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

OUTER_TEST_SIZE = 0.10
INNER_VAL_SIZE = 0.10

STRATIFY_COL = "DIAGNOSIS2"

SHARED_FEATURES = [
    "AGE",
    "APOE4COUNT",
    "MMSCORE",
    "CDGLOBAL",
]

## 2. Define external validation tasks

The same four binary classification tasks are used for shared-variable external validation.

In [ ]:
TASKS = [
    {
        "name": "AD_vs_CN",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["AD"],
        "neg_labels": ["CN"],
    },
    {
        "name": "AD_vs_MCI",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["AD"],
        "neg_labels": ["MCI"],
    },
    {
        "name": "MCI_vs_CN",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["MCI"],
        "neg_labels": ["CN"],
    },
    {
        "name": "pMCI_vs_sMCI",
        "label_col": "DIAGNOSIS2",
        "pos_labels": ["pMCI"],
        "neg_labels": ["sMCI"],
    },
]

## 3. Helper functions for shared-variable validation

These helper functions select task-specific samples and extract the shared clinical variables.

In [ ]:
def assert_binary(y, where=""):
    classes = np.unique(y)

    if len(classes) < 2:
        raise ValueError(f"{where}: only one class present: {classes}")


def map_labels(df: pd.DataFrame, label_col: str, pos_labels, neg_labels):
    labels = df[label_col].astype(str).str.strip()

    pos_set = {str(x).strip() for x in pos_labels}
    neg_set = {str(x).strip() for x in neg_labels}

    mask = labels.isin(pos_set.union(neg_set))
    sub_df = df.loc[mask].copy().reset_index(drop=True)

    y = sub_df[label_col].astype(str).str.strip().isin(pos_set).astype(int).values

    return sub_df, y


def select_shared_features(df: pd.DataFrame, feature_cols):
    missing_cols = [col for col in feature_cols if col not in df.columns]

    if len(missing_cols) > 0:
        raise ValueError(f"Missing shared feature columns: {missing_cols}")

    X = df[feature_cols].copy()

    for col in feature_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    X = X.fillna(0.0)

    return X.values.astype(np.float32)

## 4. Load ADNI and AIBL data

The ADNI dataset is used for training and internal shared-variable testing.  
The AIBL dataset is used only for external validation.

In [ ]:
df_adni = pd.read_csv(ADNI_CSV).copy()
df_aibl = pd.read_csv(AIBL_CSV).copy()


df_adni = df_adni[df_adni[STRATIFY_COL].notna()].copy()
df_adni[STRATIFY_COL] = df_adni[STRATIFY_COL].astype(str).str.strip()
df_adni = df_adni[df_adni[STRATIFY_COL] != ""].copy()

df_aibl = df_aibl[df_aibl[STRATIFY_COL].notna()].copy()
df_aibl[STRATIFY_COL] = df_aibl[STRATIFY_COL].astype(str).str.strip()
df_aibl = df_aibl[df_aibl[STRATIFY_COL] != ""].copy()

print("ADNI shape:", df_adni.shape)
print("AIBL shape:", df_aibl.shape)

print("\nADNI diagnosis distribution:")
print(df_adni[STRATIFY_COL].value_counts())

print("\nAIBL diagnosis distribution:")
print(df_aibl[STRATIFY_COL].value_counts())

## 5. Create ADNI train/test split

The ADNI cohort is split into training and held-out test sets.  
AIBL is not included in this split.

In [ ]:
df_adni_train, df_adni_test = train_test_split(
    df_adni,
    test_size=OUTER_TEST_SIZE,
    random_state=GLOBAL_SEED,
    stratify=df_adni[STRATIFY_COL].values,
)

df_adni_train = df_adni_train.reset_index(drop=True)
df_adni_test = df_adni_test.reset_index(drop=True)

print("ADNI train samples:", len(df_adni_train))
print("ADNI test samples:", len(df_adni_test))

## 6. Train ADNI shared-variable model and evaluate on AIBL

For each task, the model is trained using only ADNI data and shared clinical variables.  
The selected model is then evaluated on the ADNI held-out test set and the AIBL external cohort.

In [ ]:
def train_and_validate_external(task, cfg, seed):
    set_seed(seed)

    task_name = task["name"]

    df_train_task, y_train_all = map_labels(
        df_adni_train,
        task["label_col"],
        task["pos_labels"],
        task["neg_labels"],
    )

    df_test_task, y_adni_test = map_labels(
        df_adni_test,
        task["label_col"],
        task["pos_labels"],
        task["neg_labels"],
    )

    df_aibl_task, y_aibl = map_labels(
        df_aibl,
        task["label_col"],
        task["pos_labels"],
        task["neg_labels"],
    )

    assert_binary(y_train_all, f"{task_name} ADNI train")
    assert_binary(y_adni_test, f"{task_name} ADNI test")
    assert_binary(y_aibl, f"{task_name} AIBL external")

    X_train_all = select_shared_features(df_train_task, SHARED_FEATURES)
    X_adni_test = select_shared_features(df_test_task, SHARED_FEATURES)
    X_aibl = select_shared_features(df_aibl_task, SHARED_FEATURES)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_all,
        y_train_all,
        test_size=INNER_VAL_SIZE,
        stratify=y_train_all,
        random_state=seed,
    )

    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_adni_test_scaled = scaler.transform(X_adni_test)
    X_aibl_scaled = scaler.transform(X_aibl)

    batch_size = int(cfg["batch_size"])

    train_loader = make_loader(
        VectorDataset(X_tr_scaled, y_tr),
        batch_size=batch_size,
        shuffle=True,
        seed=seed,
    )

    val_loader = make_loader(
        VectorDataset(X_val_scaled, y_val),
        batch_size=batch_size,
        shuffle=False,
        seed=seed,
    )

    adni_test_loader = make_loader(
        VectorDataset(X_adni_test_scaled, y_adni_test),
        batch_size=batch_size,
        shuffle=False,
        seed=seed,
    )

    aibl_loader = make_loader(
        VectorDataset(X_aibl_scaled, y_aibl),
        batch_size=batch_size,
        shuffle=False,
        seed=seed,
    )

    model = SAEFirstAttnClassifier(
        n_features=X_tr_scaled.shape[1],
        d_token=int(cfg["d_model"]),
        latent_dim=int(cfg["latent_dim"]),
        n_heads=int(cfg["n_heads"]),
        n_layers=int(cfg["n_layers"]),
        ff_mult=int(cfg["ff_mult"]),
        attn_dropout=float(cfg["attn_dropout"]),
        use_cls=bool(cfg["use_cls"]),
        sae_enc_hidden=tuple(cfg["enc_hidden"]),
        sae_dec_hidden=tuple(cfg["dec_hidden"]),
        clf_hidden=tuple(cfg["clf_hidden"]),
        mlp_dropout=float(cfg["mlp_dropout"]),
    ).to(device)

    ce_loss = get_class_weighted_ce(y_tr)
    mse_loss = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
    )

    lambda_recon = float(cfg["lambda_recon"])

    best_auc = -np.inf
    best_state = None
    best_epoch = None
    patience_count = 0

    for epoch in range(1, EPOCHS_FINAL + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits, recon = model(xb)

            loss_cls = ce_loss(logits, yb)
            loss_rec = mse_loss(recon, xb)
            loss = loss_cls + lambda_recon * loss_rec

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        y_val_true, y_val_prob, _ = predict_on_loader(model, val_loader)

        try:
            val_auc = roc_auc_score(y_val_true, y_val_prob)
        except Exception:
            val_auc = np.nan

        improved = (not np.isnan(val_auc)) and (val_auc > best_auc + EARLY_STOP_MIN_DELTA)

        if improved:
            best_auc = float(val_auc)
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_count = 0
        else:
            patience_count += 1

            if patience_count >= EARLY_STOP_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_adni_true, y_adni_prob, y_adni_pred = predict_on_loader(model, adni_test_loader)
    y_aibl_true, y_aibl_prob, y_aibl_pred = predict_on_loader(model, aibl_loader)

    adni_metrics = compute_metrics(y_adni_true, y_adni_prob, y_adni_pred)
    aibl_metrics = compute_metrics(y_aibl_true, y_aibl_prob, y_aibl_pred)

    return {
        "task": task_name,
        "n_adni_train": len(y_train_all),
        "n_adni_test": len(y_adni_test),
        "n_aibl": len(y_aibl),
        "best_val_auc": best_auc,
        "best_epoch": best_epoch,
        "ADNI_AUC": adni_metrics["AUC"],
        "ADNI_ACC": adni_metrics["ACC"],
        "ADNI_PRE": adni_metrics["PRE"],
        "ADNI_SEN": adni_metrics["SEN"],
        "ADNI_SPE": adni_metrics["SPE"],
        "ADNI_F1": adni_metrics["F1"],
        "AIBL_AUC": aibl_metrics["AUC"],
        "AIBL_ACC": aibl_metrics["ACC"],
        "AIBL_PRE": aibl_metrics["PRE"],
        "AIBL_SEN": aibl_metrics["SEN"],
        "AIBL_SPE": aibl_metrics["SPE"],
        "AIBL_F1": aibl_metrics["F1"],
    }

## 7. Run external validation

The following cell runs shared-variable ADNI training and AIBL external validation for all four tasks.

In [ ]:
external_results = []

for task in TASKS:
    print(f"\nRunning external validation task: {task['name']}")

    result = train_and_validate_external(
        task=task,
        cfg=BEST_CFG,
        seed=GLOBAL_SEED,
    )

    external_results.append(result)

    print(
        f"{task['name']} | "
        f"ADNI AUC: {result['ADNI_AUC']:.4f} | "
        f"AIBL AUC: {result['AIBL_AUC']:.4f}"
    )

external_results_df = pd.DataFrame(external_results)
external_results_df

## 8. Save external validation results

The shared-variable ADNI/AIBL validation results are saved for manuscript reporting.

In [ ]:
external_results_path = f"{RESULTS_DIR}/external_aibl_results.csv"
external_results_df.to_csv(external_results_path, index=False)

print(f"Saved external validation results to: {external_results_path}")

## 9. Manuscript-style summary table

This table summarizes the main shared-variable validation results used in the manuscript.

In [ ]:
summary_table = external_results_df[
    [
        "task",
        "ADNI_AUC",
        "ADNI_ACC",
        "AIBL_AUC",
        "AIBL_ACC",
    ]
].copy()

summary_table

summary_table_path = f"{RESULTS_DIR}/shared_variable_validation_summary.csv"
summary_table.to_csv(summary_table_path, index=False)

print(f"Saved summary table to: {summary_table_path}")

## 10. Expected outputs

After running this notebook, the following files should be generated locally:

```text
results/external_aibl_results.csv
results/shared_variable_validation_summary.csv